In this notebook there is the code needed for
 * Gaussian Mixture analysis
 * Figure 5.1: cost vs rounds for 10% of KDDcup1999
 * Figure 5.2: cost vs rounds for Gaussian Mixture

## 1. Import e flag

In [27]:
import os, time
import numpy as np
import pandas as pd

from src.launch_cluster import launch_cluster, shutdown_cluster
from src.data_loader import load_dataset, make_gauss_mixture
from src.paper_experiments import (
    run_fig51, run_fig52, run_table34,
    plot_fig51, plot_fig52, table34_cost_table, table34_time_table,
)
from src.benchmark import RESULTS_DIR

# --- execution flags ---
RESTART_CLUSTER = True   # reduced search
RUN_FIG52_TINY   = False   
RUN_FIG52_FULL   = False   
RUN_CLUSTER      = True   
RUN_FIG51        = False  
RUN_TABLE34      = False   # KDD full

SEED = 42
N_RUNS = 11                # as in paper: median over 11 run

In [28]:
# If cluster alredy up, we shut it down
if RESTART_CLUSTER:
    
    from dask.distributed import Client
    
    # connecting to alredy active scheduler (HEAD_IP:SCHEDULER_PORT)
    client = Client("10.67.22.194:8786")
    
    client.shutdown()   # cluster shutdown
    print(" Cluster shutdown completed ")
else:
    print("RESTART_CLUSTER = False")

 Cluster shutdown completed 


## 2. Fig 5.2 — GaussMixture 

Start run for gaussian mixture for R ∈ {1,10,100} (variance), ℓ/k ∈ {0.1,…,10},
r = 0..15, k=50)

In [23]:
if RUN_FIG52_FULL:
    df52 = run_fig52(client=None, seed=SEED, n_runs=N_RUNS)
    _ts = time.strftime("%Y%m%d_%H%M%S")
    _csv = os.path.join(RESULTS_DIR, f"fig52_{_ts}.csv")
    df52.to_csv(_csv, index=False)
    print("Saved", _csv)
    plot_fig52(df52, output_dir="figures")
else:
    print("RUN_FIG52_FULL = False")

RUN_FIG52_FULL = False


## 3. Loading KDD

10% dataset -> `DATASET_URL_10PC`

100% -> `DATASET_URL_FULL`

In [29]:
DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
DATASET_URL_FULL = "https://ndownloader.figshare.com/files/5976045"

RAW_GZ_PATH  = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz"
PARQUET_PATH = "/tmp/kddcup_data.parquet"

RAW_GZ_PATH_10PC   = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz" # compressed (.gz) dataset file
PARQUET_PATH_10PC = '/tmp/kddcup_data.parquet'

COL_NAMES = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell",
    "su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label",
]

### Cluster on/off

In [30]:
# DO NOT RUN if already existing!
if RUN_CLUSTER:
    N_WORKERS = 8
    NUM_PARTITIONS = 8 * N_WORKERS
    cluster, client = launch_cluster(N_WORKERS)
else:
    print("RUN_CLUSTER = False")

Initializing the SSH cluster with 8 workers...
Selected workers: ['10.67.22.254', '10.67.22.34', '10.67.22.145', '10.67.22.121', '10.67.22.192', '10.67.22.18', '10.67.22.187', '10.67.22.48']


2026-09-12 15:59:40,506 - distributed.deploy.ssh - INFO - 2026-09-12 15:59:40,505 - distributed.http.proxy - INFO - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-09-12 15:59:40,535 - distributed.deploy.ssh - INFO - 2026-09-12 15:59:40,534 - distributed.scheduler - INFO - State start
2026-09-12 15:59:40,538 - distributed.deploy.ssh - INFO - 2026-09-12 15:59:40,538 - distributed.scheduler - INFO -   Scheduler at:   tcp://10.67.22.194:8786
2026-09-12 15:59:42,404 - distributed.deploy.ssh - INFO - 2026-09-12 15:59:42,403 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.254:35437'
2026-09-12 15:59:42,463 - distributed.deploy.ssh - INFO - 2026-09-12 15:59:42,461 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.48:46455'
2026-09-12 15:59:42,523 - distributed.deploy.ssh - INFO - 2026-09-12 15:59:42,521 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.121

Cluster started and connection established successfully!



## 4. Fig 5.1 — KDD 10%, exact-ℓ 

k ∈ {17,33,65,129}, ℓ/k ∈ {1,2,4}, r = 1..10

In [31]:
X_bag_10_percent, (mean_ar, std_ar)=load_dataset(n_partitions=NUM_PARTITIONS,
                   client=client,
                   dataset_url=DATASET_URL_10PC,
                   raw_gz_path=RAW_GZ_PATH_10PC,
                   parquet_path=PARQUET_PATH_10PC,
                   parquet_path_workers=PARQUET_PATH_10PC,
                   col_names=COL_NAMES,
                   force_download=False)

TypeError: load_dataset() got an unexpected keyword argument 'parquet_path_workers'

In [ ]:
if RUN_CLUSTER and RUN_FIG51:
    #X_bag, _ = load_dataset(DATASET_URL_10PC, RAW_GZ_PATH, PARQUET_PATH,
         #                   PARQUET_PATH, COL_NAMES,
          #                  n_partitions=NUM_PARTITIONS, client=client)
    #df51 = run_fig51(client, X_bag, seed=SEED, n_runs=N_RUNS,
     #                num_partitions=NUM_PARTITIONS)
    df51.to_csv(os.path.join(RESULTS_DIR, "fig51_full.csv"), index=False)
    plot_fig51(df51, output_dir="figures")
else:
    print("RUN_CLUSTER/RUN_FIG51 = False")

Using cached dataset: /home/ubuntu/Project/libero_development/data/kddcup_data.gz
Converting .gz -> Parquet chunk sizes...
Parquet file created (compressed with snappy).
Number of partitions before preprocessing: 64
Constant columns: ['num_outbound_cmds', 'is_host_login']
Computing global mean and std (first pass over data)...
Distributed bag created with 32 partitions.
Number of samples: 494021


/home/ubuntu/pyvenv/lib/python3.10/site-packages/sklearn/base.py:1365: ConvergenceWarning: Number of distinct clusters (13) found smaller than n_clusters (17). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:729: UserWarning: 4 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(


[fig51] k=17, l=17 (l/k=1), r=1: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=2: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=3: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=4: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=5: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=6: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=7: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=8: 11 run completate
[fig51] k=17, l=17 (l/k=1), r=9: 11 run completate


In [4]:
if RUN_CLUSTER and RUN_FIG51:
    df51 = pd.read_csv("/home/ubuntu/Project/libero_development/results/with_old_code/other_old/fig51_full.csv")  # <-- il path che hai trovato
    plot_fig51(df51, output_dir="figures")
else:
    print("RUN_CLUSTER/RUN_FIG51 = False")

Saved figures/fig51_cost_vs_rounds.png


## Cost/Time vs ℓ/k — KDD 10%, r=5  

ℓ/k ∈ {0.1, 0.5, 1, 2}, k ∈ {500, 1000},
mean ± std over N_RUNS repetitions.


In [ ]:
from src.paper_experiments import run_l_sweep, plot_l_sweep

PARQUET_PATH_LK = "/tmp/kddcup_data_lk.parquet"
if RUN_CLUSTER:
    X_bag_10pct, _ = load_dataset(DATASET_URL_10PC, RAW_GZ_PATH, PARQUET_PATH_LK,
                                   PARQUET_PATH_LK, COL_NAMES,
                                   n_partitions=NUM_PARTITIONS, client=client)

    df_lk = run_l_sweep(client, X_bag_10pct,
                        k_values=(500, 1000),
                        l_over_k_values=(10,20,100),
                        r_fixed=5, n_runs=N_RUNS, seed=SEED,
                        num_partitions=NUM_PARTITIONS,
                        policy="auto")
    df_lk.to_csv(os.path.join(RESULTS_DIR, "l_sweep_10pct_extra.csv"), index=False)
    #plot_l_sweep(df_lk, output_dir="figures")
else:
    print("RUN_CLUSTER = False")


In [6]:
df_lk = pd.read_csv(os.path.join(RESULTS_DIR, "l_sweep_10pct.csv"))
df_lk_clean = df_lk[df_lk["l_over_k"] != 0.2]
plot_l_sweep(df_lk_clean, output_dir="figures")

df_lk_clean.to_csv(os.path.join(RESULTS_DIR, "l_sweep_10pct_no02.csv"), index=False)


Saved figures/l_sweep_k500.png
Saved figures/l_sweep_k1000.png


In [10]:
df_base = pd.read_csv(os.path.join(RESULTS_DIR, "l_sweep_10pct_no02.csv"))
df_extra = pd.read_csv(os.path.join(RESULTS_DIR, "l_sweep_10pct_extra.csv"))

df_all = (pd.concat([df_base, df_extra], ignore_index=True)
            .sort_values(["k", "l_over_k", "run"])
            .reset_index(drop=True))

df_all.to_csv(os.path.join(RESULTS_DIR, "l_sweep_10pct_full.csv"), index=False)
print(f"{len(df_base)} + {len(df_extra)} -> {len(df_all)} rows")
print(df_all.groupby(["k", "l_over_k"]).size())

110 + 66 -> 176 rows
k     l_over_k
500   0.1         11
      0.5         11
      1.0         11
      2.0         11
      5.0         11
      10.0        11
      20.0        11
      100.0       11
1000  0.1         11
      0.5         11
      1.0         11
      2.0         11
      5.0         11
      10.0        11
      20.0        11
      100.0       11
dtype: int64


In [17]:
plot_l_sweep_log(df_all, output_dir="figures",logx=True)

Saved figures/l_sweep_k500.png
Saved figures/l_sweep_k1000.png


['figures/l_sweep_k500.png', 'figures/l_sweep_k1000.png']

In [16]:
import matplotlib.pyplot as plt
def plot_l_sweep_log(results, output_dir="figures", dpi=150, logx=False, logy=False):
    """One figure per k: final cost (left) and total running time (right)
    vs l/k, mean +/- std over the runs. ``results`` is a DataFrame or a CSV
    path (as produced by ``run_l_sweep``). ``logx``/``logy`` set log scale on
    the respective axes (useful when l/k spans decades).
    Returns the list of saved paths."""
    df = _load_csv(results) if isinstance(results, str) else results.copy()
    if "artifact" in df.columns:
        df = df[df["artifact"] == "l_sweep"]
    if "failed" in df.columns:
        df = df[df["failed"] == False]  # noqa: E712 (pandas mask)
    if "time_total" not in df.columns:
        df = df.assign(time_total=df["time_seed"] + df["time_fit"])

    metrics = [("cost_final", "Cost"), ("time_total", "Time (s)")]
    colors = {"cost_final": "tab:blue", "time_total": "tab:orange"}
    outpaths = []
    for k in sorted(df["k"].unique()):
        g = (df[df["k"] == k]
             .groupby("l_over_k")[["cost_final", "time_total"]]
             .agg(["mean", "std"]))
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
        for ax, (col, ylab) in zip(axes, metrics):
            ax.errorbar(g.index, g[(col, "mean")], yerr=g[(col, "std")],
                        marker="o", capsize=4, color=colors[col])
            ax.set_xlabel(r"$\ell\,/\,k$")
            ax.set_ylabel(f"{ylab} (mean $\\pm$ std)")
            ax.set_title(f"{ylab.split(' (')[0]} vs $\\ell/k$  ($k={k}$)")
            if logx:
                ax.set_xscale("log")
            if logy:
                ax.set_yscale("log")
            ax.grid(True, alpha=0.3, which="both")
        fig.suptitle(f"Cost and running time vs oversampling factor "
                     f"$\\ell/k$  ($k = {k}$)", fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        outpath = os.path.join(output_dir, f"l_sweep_k{k}.png")
        fig.savefig(outpath, dpi=dpi)
        plt.close(fig)
        print("Saved", outpath)
        outpaths.append(outpath)
    return outpaths


# Cluster shutdown

In [ ]:
# Da eseguire a fine lavoro
shutdown_cluster(cluster, client)